# 动手练习：生物分子网络与拓扑特征

来自《生物医学网络科学》（Biomedical Network Science）。

每个任务先运行“准备”格，再在“你的代码”格中完成 TODO，最后运行“核对”格：打印 ✓ 表示通过，否则会给出提示。数据文件放在与本笔记本同级的 `data/` 文件夹中。

In [ ]:
# 数据文件（JupyterLite 专用；文件已在时什么也不做）
import os, js
from pyodide.http import pyfetch
_base = str(js.location.href).split("/extensions/")[0].split("/build/")[0] + "/files/practice/data/"
os.makedirs("data", exist_ok=True)
for _f in ['yeast_genes_essentiality.csv', 'yeast_string_v12_experimental.csv']:
    if not os.path.exists("data/" + _f):
        _r = await pyfetch(_base + _f)
        open("data/" + _f, "wb").write(await _r.bytes())


## 任务 1：重尾度分布与“Jeong 式”统计

本练习前 5 个任务中的“蛋白网络”“调控网络”和表达曲线都是程序生成的**模拟数据**，只用于演示方法，不对应任何真实物种；第 6 个任务使用真实的酵母数据。

setup 生成一个 1500 节点的 BA 网络 `G_ba`（模拟“蛋白相互作用网络”）和同规模、同边数的 ER 网络 `G_er`。

1. 写函数 `ccdf(G)`，返回 `(ks, P)`：`ks` 为排序后的不同度值，`P[i]` = 度 ≥ `ks[i]` 的节点比例；
2. 仿照 Jeong 等（2001），计算 BA 网络中度 ≤ 5 的节点比例 `low_ba`、度 > 15 的节点比例 `high_ba`，以及 ER 网络中度 > 15 的比例 `high_er`；
3. 在双对数坐标上画两条 CCDF。

期望看到：BA 网络的 CCDF 呈长尾，存在度上百的枢纽；ER 网络没有度 > 15 的节点。

In [ ]:
# 准备（直接运行）
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

G_ba = nx.barabasi_albert_graph(1500, 2, seed=1)
G_er = nx.gnm_random_graph(1500, G_ba.number_of_edges(), seed=1)

In [ ]:
def ccdf(G):
    # TODO
    return None, None


low_ba, high_ba, high_er = None, None, None

In [ ]:
# 核对（不用修改）
ks_chk, P_chk = ccdf(G_ba)
assert ks_chk is not None and P_chk is not None, "ccdf 还没有返回结果"
assert abs(P_chk[0] - 1) < 1e-12 and np.all(np.diff(P_chk) <= 0), "CCDF 从 1 开始且单调不增：P[i] 是度 ≥ ks[i] 的比例"
assert ks_chk[-1] == 105 and abs(P_chk[-1] - 1 / 1500) < 1e-12, "最大度 105 处 CCDF 应为 1/1500"
assert abs(low_ba - 0.857) < 0.002 and abs(high_ba - 0.025) < 0.002, "BA 网络：度 ≤ 5 约占 0.857，度 > 15 约占 0.025"
assert high_er == 0, "ER 网络中不应有度 > 15 的节点"
print("✓ 任务 1 通过")

## 任务 2：删除枢纽 vs 随机删除

写函数 `giant_fraction_after_removal(G, order, f)`：复制 `G`，删除 `order` 中前 `int(f * N)` 个节点，返回最大连通分量节点数 / 原网络节点数 $N$。按度从高到低（`order_hub`）和随机顺序（`order_rand`）分别删除 5%、10%、20% 的节点，结果存入列表 `res_hub`、`res_rand`。

期望看到：随机删除时巨分支缓慢缩小；删除枢纽时巨分支迅速崩塌。思考：这能否解释“枢纽更可能是必需的”？（提示：专题页“致死性与中心性”一节的争议——细胞功能并不需要全网连通。）

In [ ]:
# 准备（直接运行）
rng2 = np.random.default_rng(0)
order_hub = [n for n, _ in sorted(G_ba.degree(), key=lambda x: -x[1])]
order_rand = list(rng2.permutation(list(G_ba.nodes())))
fracs = [0.05, 0.10, 0.20]

In [ ]:
def giant_fraction_after_removal(G, order, f):
    # TODO
    return None


res_hub, res_rand = [], []

In [ ]:
# 核对（不用修改）
assert giant_fraction_after_removal(G_ba, order_hub, 0) == 1.0, "不删除任何节点时巨分支比例应为 1"
assert len(res_hub) == 3 and len(res_rand) == 3, "每种策略应有 3 个结果"
assert res_hub[-1] < 0.1, "删除 20% 枢纽后巨分支应几乎崩塌（< 0.1）；分母应是原网络节点数"
assert res_rand[-1] > 0.7, "随机删除 20% 节点后巨分支应仍保留大部分"
assert all(h < r for h, r in zip(res_hub, res_rand)), "同样比例下，攻击枢纽的破坏应大于随机删除"
print("✓ 任务 2 通过")

## 任务 3：数前馈环，并与保度随机化网络比较

setup 构造一个 200 个节点的有向“调控网络” `D`：随机有向边 + 人为嵌入的 30 个前馈环（X→Y、Y→Z、X→Z）。

1. 写函数 `count_ffl(D)`：统计满足 X→Y、Y→Z、X→Z 的三元组数；
2. 用 setup 中保持每个节点入度和出度的边交换函数 `degree_preserving_shuffle` 生成 30 个随机化网络（用 `rng3`），计算 FFL 数的均值与标准差，得到 Z 分数 `z_ffl`。

期望看到：真实网络的 FFL 数远多于随机化网络，Z 分数很大。

> **提示：** 对每条边 (x, y)，`set(D.successors(x)) & set(D.successors(y))` 就是可以作为 Z 的节点（去掉 x、y 本身）。

In [ ]:
# 准备（直接运行）
def make_trn(seed=0):
    """模拟数据，仅用于演示方法：随机有向网络 + 嵌入的前馈环。"""
    rng = np.random.default_rng(seed)
    D = nx.gnm_random_graph(200, 300, directed=True, seed=seed)
    for _ in range(30):
        x, y, z = rng.choice(200, 3, replace=False)
        D.add_edges_from([(x, y), (y, z), (x, z)])
    D.remove_edges_from(list(nx.selfloop_edges(D)))
    return D


def degree_preserving_shuffle(D, n_swap, rng):
    """有向边交换：(a→b, c→d) 换成 (a→d, c→b)，保持所有节点的入度和出度。"""
    D = D.copy()
    edges = list(D.edges())
    done = 0
    while done < n_swap:
        i, j = rng.integers(len(edges), size=2)
        (a, b), (c, d) = edges[i], edges[j]
        if len({a, b, c, d}) < 4 or D.has_edge(a, d) or D.has_edge(c, b):
            continue
        D.remove_edges_from([(a, b), (c, d)])
        D.add_edges_from([(a, d), (c, b)])
        edges[i], edges[j] = (a, d), (c, b)
        done += 1
    return D


D = make_trn()
rng3 = np.random.default_rng(1)

In [ ]:
def count_ffl(D):
    # TODO
    return None


z_ffl = None

In [ ]:
# 核对（不用修改）
assert count_ffl(nx.DiGraph([(1, 2), (2, 3), (1, 3)])) == 1, "单个前馈环应计为 1"
assert count_ffl(nx.DiGraph([(1, 2), (2, 3), (3, 1)])) == 0, "三节点反馈环不是前馈环，应计为 0"
assert count_ffl(D) == 37, "模拟网络中应有 37 个前馈环（30 个嵌入的 + 随机出现的）"
assert z_ffl is not None and z_ffl > 4, "Z 分数应很大（约 7.5）；随机化网络必须保持入度和出度"
print("✓ 任务 3 通过")

## 任务 4：一致型前馈环（与门）的持续性检测

用欧拉法模拟两种线路对输入 X 脉冲的响应（步长 0.01，总时长 15，脉冲从 t = 1 开始）：

- 简单调控：$dZ/dt=\beta X-\alpha Z$；
- 一致型 1 型 FFL（与门）：$dY/dt=\beta X-\alpha Y$，$dZ/dt=\beta X\cdot\theta(Y>K_{YZ})-\alpha Z$。

补全 `simulate` 中 Z 的更新式，分别给短脉冲（持续 0.5）和长脉冲（持续 4），记录四个峰值 `peak_simple_short`、`peak_ffl_short`、`peak_simple_long`、`peak_ffl_long`，以及长脉冲下 FFL 的 Z 首次超过 0.01 的时刻 `t_on_ffl`。

期望看到：简单调控对两种脉冲都响应；FFL 对短脉冲完全不响应、对长脉冲在约 0.7 的延迟后才响应——过滤短暂噪声。Y 从 0 涨到 $K_{YZ}=0.5$（稳态为 1）需要 $\ln 2/\alpha\approx0.69$。

In [ ]:
def simulate(pulse_len, ffl=True, beta=1.0, alpha=1.0, K_yz=0.5, dt=0.01, T=15.0, t_on=1.0):
    t = np.arange(0, T, dt)
    X = ((t >= t_on) & (t < t_on + pulse_len)).astype(float)
    Y = np.zeros_like(t)
    Z = np.zeros_like(t)
    for i in range(1, len(t)):
        Y[i] = Y[i - 1] + dt * (beta * X[i - 1] - alpha * Y[i - 1])
        Z[i] = Z[i - 1]   # TODO: 写出 Z 的更新式（ffl=True 时加上与门）
    return t, X, Y, Z


peak_simple_short, peak_ffl_short = None, None
peak_simple_long, peak_ffl_long = None, None
t_on_ffl = None

In [ ]:
# 核对（不用修改）
assert peak_simple_short is not None and peak_ffl_long is not None, "四个峰值还没有计算"
assert abs(peak_simple_short - 0.395) < 0.01, "简单调控对 0.5 的短脉冲应有约 0.39 的峰值"
assert peak_ffl_short < 1e-9, "与门 FFL 对短脉冲不应响应：Y 还没积累到 K_YZ，脉冲就结束了"
assert peak_simple_long > 0.95 and peak_ffl_long > 0.9, "长脉冲下两种线路都应接近稳态 1"
assert t_on_ffl is not None and 1.6 < t_on_ffl < 1.8, "FFL 应在输入开始（t = 1）后约 0.7 才开启"
print("✓ 任务 4 通过")

## 任务 5：度与必需性的混杂：复合物

setup 构造一个模拟网络：1000 个“散在”蛋白，加上 40 个大小 6–12 的“复合物”（内部以 0.7 的概率两两相连），再随机加 2500 条背景边。必需性**只由是否属于复合物决定**：复合物成员以 0.6 的概率必需，其余蛋白以 0.1 的概率必需——与度本身无关。

计算必需蛋白与非必需蛋白的平均度之差：全部蛋白 `diff_all`、只看复合物成员 `diff_in`、只看散在蛋白 `diff_out`；再计算度最高 10% 的蛋白中的必需比例 `ess_top10`。

期望看到：单看全部蛋白，必需蛋白的平均度明显更高，“度最高的蛋白更可能必需”；按是否属于复合物分层后，差异基本消失。度与必需性的相关可以完全来自它们共同的原因——所在的模块（专题页“必需复合物/模块”解释）。

In [ ]:
# 准备（直接运行）
rng5 = np.random.default_rng(7)
Gc = nx.Graph()
Gc.add_nodes_from(range(1000))
complex_id = np.full(1000, -1)
next_id = 1000
for c in range(40):
    size = int(rng5.integers(6, 13))
    members = list(range(next_id, next_id + size))
    next_id += size
    Gc.add_nodes_from(members)
    complex_id = np.r_[complex_id, [c] * size]
    for i in members:
        for j in members:
            if i < j and rng5.random() < 0.7:
                Gc.add_edge(i, j)
Nc = Gc.number_of_nodes()
for _ in range(2500):
    u, v = rng5.integers(Nc, size=2)
    if u != v:
        Gc.add_edge(int(u), int(v))
in_complex = complex_id >= 0
essential = np.where(in_complex, rng5.random(Nc) < 0.6, rng5.random(Nc) < 0.1).astype(int)
deg_c = np.array([Gc.degree(i) for i in range(Nc)])

In [ ]:
diff_all, diff_in, diff_out = None, None, None
ess_top10 = None

In [ ]:
# 核对（不用修改）
assert diff_all is not None and diff_all > 2.5, "全部蛋白中，必需蛋白的平均度应明显更高（差约 3.4）"
assert diff_in is not None and abs(diff_in) < 1 and abs(diff_out) < 1, "按是否属于复合物分层后，平均度差应基本消失"
assert ess_top10 is not None and ess_top10 > 0.4, "度最高的 10% 蛋白中必需比例应明显高于整体"
print("✓ 任务 5 通过")

## 任务 6：小分析：真实酵母网络中的中心性–致死性

用真实数据：`data/yeast_string_v12_experimental.csv`（STRING v12.0 酵母实验证据通道，CC BY 4.0）与 `data/yeast_genes_essentiality.csv`（SGD 注释的 Giaever 等 2002 缺失实验结果，CC BY 4.0；1 = 必需，0 = 非必需，空 = 未注释）。出处见本讲真实数据案例与 `data/SOURCES.md`。

选一个证据阈值 `threshold`（400–900），只保留 experiments ≥ 阈值的边构网；对有必需性注释的蛋白，计算度最高 5% 的蛋白中的必需比例 `ess_hub` 与其余蛋白中的必需比例 `ess_rest`。

换几个阈值比较：中心性–致死性规律稳健吗？阈值越高，网络越小、哪类蛋白更容易留下来？你在任务 5 中看到的混杂，在真实数据中可能来自什么（复合物、研究热度、实验方法）？

In [ ]:
# 准备（直接运行）
import pandas as pd

ppi = pd.read_csv("data/yeast_string_v12_experimental.csv")
yg = pd.read_csv("data/yeast_genes_essentiality.csv")
ess_map = dict(zip(yg.orf, yg.essential))

In [ ]:
threshold = 700
ess_hub, ess_rest = None, None

In [ ]:
# 核对（不用修改）
assert 400 <= threshold <= 900, "threshold 请取 400–900"
assert ess_hub is not None and ess_rest is not None, "ess_hub、ess_rest 还没有计算"
assert 0 <= ess_hub <= 1 and 0 <= ess_rest <= 1, "必需比例应在 0 到 1 之间"
print("✓ 任务 6 通过")